# Expérience 04 Continual pretraining : tokenizer wolof 4 000 + Full Fine-Tuning

**Modèle :** `SmolLM3-3B-Base` · **Données :** wolof monolingue · **Méthode :** full fine-tuning bfloat16

## Pourquoi le vocabulaire 4 000

L'ablation menée en LoRA a comparé les vocabulaires 4 000 et 8 000 dans des conditions identiques :

| Vocabulaire | BPB avant | BPB après | Écart |
|---|---|---|---|
| 4 000 | 2,0110 | **0,8767** | — |
| 8 000 | 2,0721 | 0,8815 | +0,0048 |

L'écart initial (0,068 en faveur du 4 000) tombe à 0,005 après entraînement : le désavantage
du 8 000 venait d'une initialisation imparfaite de ses tokens rares, corrigée par l'entraînement.
Les deux configurations étant équivalentes, le **4 000** est retenu — même performance,
deux fois moins d'embeddings à entraîner.

> **Hypothèse assumée :** cette équivalence, établie en LoRA, est supposée valoir aussi en
> full fine-tuning. Elle n'a pas été vérifiée directement, faute de budget de calcul.




## 1  Environnement

In [ ]:
!pip install -q -U accelerate datasets pandas matplotlib transformers bitsandbytes

## 2  Configuration


In [ ]:
ROOT       = "/content/drive/MyDrive/SmolLM3-3B-Base"
BASE_ID    = "HuggingFaceTB/SmolLM3-3B-Base"

RUN        = "wolof-ext4000-full"
TOK_DIR    = f"{ROOT}/tokenizers/smollm3_wolof_ext_4000"
MODEL_DIR  = f"{ROOT}/models/smollm3-wolof-ext4000-init"
OUT        = f"{ROOT}/runs/{RUN}"

MAX_STEPS  = 250         
SEQ_LEN    = 1024
MICRO_BS   = 4
GRAD_ACC   = 8           
LR         = 2e-5         
SEED       = 42
OPTIM      = "adamw_torch"        

TOKENS_PAR_PAS = MICRO_BS * GRAD_ACC * SEQ_LEN
print(f"{TOKENS_PAR_PAS:,} tokens/pas | {MAX_STEPS} pas = {MAX_STEPS*TOKENS_PAR_PAS/1e6:.1f} M tokens")

In [ ]:
import os, torch
for d in (TOK_DIR, MODEL_DIR):
    assert os.path.isdir(d), f"introuvable : {d}"
os.makedirs(OUT, exist_ok=True)

libre, total = torch.cuda.mem_get_info()
print(f"GPU : {torch.cuda.get_device_name(0)} | {libre/1e9:.1f} / {total/1e9:.1f} Go libres")
if total/1e9 < 35:
    print("⚠️  Moins de 35 Go : envisager OPTIM = 'adamw_bnb_8bit'")

## 3  Chargement du modèle initialisé


In [ ]:
import time, math, json, gc
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(TOK_DIR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.bfloat16, device_map={"": 0})
model.config.use_cache = False

# contrôles de cohérence
assert model.config.vocab_size == len(tok), \
    f"incohérence : modèle {model.config.vocab_size} vs tokenizer {len(tok)}"
eos = model.config.eos_token_id
eos = eos[0] if isinstance(eos, list) else eos
print(f"vocabulaire : {len(tok):,} | eos_id={eos} → {tok.decode([eos])!r}")

model.gradient_checkpointing_enable()
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_tot   = sum(p.numel() for p in model.parameters())
print(f"paramètres entraînables : {n_train:,} / {n_tot:,} ({100*n_train/n_tot:.0f} %)")

## 4  Données


In [ ]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_parquet(f"{ROOT}/data/wolof_bpe_train_v1.parquet")
eval_df  = pd.read_parquet(f"{ROOT}/data/wolof_eval_clean_v1.parquet")

TEXTES_TRAIN = train_df["text"].astype(str).tolist()
TEXTES_EVAL  = eval_df["text"].astype(str).tolist()
print(f"train : {len(TEXTES_TRAIN):,} | éval propre : {len(TEXTES_EVAL):,}")

def paqueter(textes, tokenizer, longueur):
    ids = []
    for t in textes:
        ids.extend(tokenizer.encode(t, add_special_tokens=False) + [tokenizer.eos_token_id])
    n = (len(ids) // longueur) * longueur
    blocs = [ids[i:i+longueur] for i in range(0, n, longueur)]
    return Dataset.from_dict({"input_ids": blocs, "labels": [b[:] for b in blocs]})

ds_train = paqueter(TEXTES_TRAIN, tok, SEQ_LEN).shuffle(seed=SEED)
ds_eval  = paqueter(TEXTES_EVAL,  tok, SEQ_LEN)
print(f"blocs — train {len(ds_train):,} | éval {len(ds_eval):,}")

## 5  BPB avant entraînement


In [ ]:
@torch.no_grad()
def evaluer_bpb(model, tokenizer, textes, max_len=1024):
    model.eval()
    nats = octets = tokens = 0
    for t in textes:
        t = str(t)
        b = len(t.encode("utf-8"))
        if b == 0: continue
        ids = tokenizer.encode(t, add_special_tokens=False)[:max_len]
        if len(ids) < 2: continue
        x = torch.tensor([ids], device="cuda:0")
        logits = model(x).logits[0, :-1]
        nats   += F.cross_entropy(logits.float(), x[0, 1:], reduction="sum").item()
        octets += b
        tokens += len(ids) - 1
    model.train()
    return {"bpb": nats/(math.log(2)*octets), "ppl": math.exp(nats/tokens),
            "n_octets": octets, "n_tokens": tokens}

bpb_avant = evaluer_bpb(model, tok, TEXTES_EVAL[:500])
print(f"BPB avant : {bpb_avant['bpb']:.4f} | PPL : {bpb_avant['ppl']:.2f}")

## 6  Entraînement

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

args = TrainingArguments(
    output_dir=OUT, max_steps=MAX_STEPS,
    per_device_train_batch_size=MICRO_BS, gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=25, eval_strategy="steps", eval_steps=50,
    save_steps=MAX_STEPS, save_total_limit=1,
    bf16=True, fp16=False, gradient_checkpointing=True,
    optim=OPTIM, report_to="none", seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train,
                  eval_dataset=ds_eval, data_collator=default_data_collator)

t0 = time.time()
res = trainer.train()
duree = (time.time() - t0) / 3600
print(res.metrics)
print(f"\nterminé en {duree:.2f} h")

## 7  Évaluation finale

In [ ]:
bpb_apres = evaluer_bpb(model, tok, TEXTES_EVAL[:500])
print(f"BPB  : {bpb_avant['bpb']:.4f} → {bpb_apres['bpb']:.4f}  "
      f"(gain {100*(1-bpb_apres['bpb']/bpb_avant['bpb']):.1f} %)")
print(f"PPL  : {bpb_avant['ppl']:.2f} → {bpb_apres['ppl']:.2f}")

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
ltr  = [(h["step"], h["loss"])      for h in hist if "loss" in h]
lval = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
st, lo = zip(*ltr)
ax[0].plot(st, lo, marker="o", ms=4, color="#1f77b4", label="train")
if lval:
    sv, lv = zip(*lval)
    ax[0].plot(sv, lv, marker="s", ms=5, color="#d62728", lw=2, label="validation")
ax[0].set_xlabel("pas"); ax[0].set_ylabel("loss")
ax[0].set_title(f"{RUN} — loss"); ax[0].legend(); ax[0].grid(alpha=.3)

lrs = [h["learning_rate"] for h in hist if "learning_rate" in h]
ax[1].plot(st[:len(lrs)], lrs, color="darkorange", lw=2)
ax[1].set_xlabel("pas"); ax[1].set_ylabel("learning rate")
ax[1].set_title("scheduler cosine"); ax[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(f"{OUT}/courbes.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"loss train : {ltr[0][1]:.3f} → {ltr[-1][1]:.3f}")
if lval: print(f"loss val   : {lval[-1][1]:.3f}")

## 8  Journal et sauvegarde


In [ ]:
journal = {
    "run": RUN, "gpu": GPU, "modele_initialise": MODEL_DIR, "tokenizer": TOK_DIR,
    "vocab": len(tok), "donnees": "wolof monolingue",
    "methode": "full fine-tuning bfloat16",
    "hyperparametres": {
        "lr": LR, "optim": OPTIM, "scheduler": "cosine", "warmup_ratio": 0.03,
        "seq_len": SEQ_LEN, "micro_bs": MICRO_BS, "grad_acc": GRAD_ACC,
        "batch_effectif": MICRO_BS*GRAD_ACC, "max_steps": MAX_STEPS, "seed": SEED,
        "gradient_checkpointing": True},
    "budget": {"tokens_vus_M": MAX_STEPS*TOKENS_PAR_PAS/1e6,
               "duree_heures": round(duree, 3),
               "epochs_equivalent": round(MAX_STEPS*MICRO_BS*GRAD_ACC/len(ds_train), 4)},
    "loss": {"train_premiere": ltr[0][1], "train_derniere": ltr[-1][1],
             "val_derniere": lval[-1][1] if lval else None,
             "moyenne_run": res.metrics.get("train_loss"),
             "courbe_train": ltr, "courbe_val": lval},
    "eval": {"split": "wolof_eval_clean_v1.parquet",
             "bpb_avant": bpb_avant["bpb"], "bpb_apres": bpb_apres["bpb"],
             "ppl_avant": bpb_avant["ppl"], "ppl_apres": bpb_apres["ppl"]},
}
json.dump(journal, open(f"{OUT}/journal.json", "w"), indent=2, ensure_ascii=False)

model.save_pretrained(f"{OUT}/model", safe_serialization=True)
tok.save_pretrained(f"{OUT}/model")
print("sauvegardé :", OUT)

In [ ]:
# libération mémoire
for nom in ["model", "trainer", "ds_train", "ds_eval"]:
    if nom in dir(): exec(f"del {nom}")
gc.collect(); torch.cuda.empty_cache()
libre, total = torch.cuda.mem_get_info()
print(f"VRAM : {libre/1e9:.1f} / {total/1e9:.1f} Go")